# Exploración de datos — BrokerHub

Notebook para probar y visualizar los datos que trae `cliente_finnhub.py` antes de insertarlos en MySQL:
- Perfil de empresa (Finnhub)
- Precio actual (Finnhub)
- Clasificación por capitalización
- Histórico diario (yfinance)

In [ ]:
import pandas as pd
from cliente_finnhub import (
    obtener_perfil_empresa,
    obtener_precio_actual,
    obtener_historico,
    clasificar_por_capitalizacion,
)

pd.set_option("display.max_columns", None)

## 1. Define la lista de tickers a probar

In [ ]:
tickers = ["AAPL", "MSFT", "NVDA", "XOM", "JPM"]

## 2. Perfil de empresa + clasificación por capitalización
Trae el perfil de cada ticker y arma una tabla resumen (esto alimenta `Emisor`, `Instrumento_Financiero` y `Categoria_Instrumento`).

In [ ]:
filas_perfil = []

for ticker in tickers:
    perfil = obtener_perfil_empresa(ticker)
    categoria = clasificar_por_capitalizacion(perfil.get("marketCapitalization"))
    filas_perfil.append({
        "ticker": ticker,
        "razon_social": perfil.get("name"),
        "sector": perfil.get("finnhubIndustry"),
        "pais": perfil.get("country"),
        "market_cap_millones": perfil.get("marketCapitalization"),
        "categoria_nivel3": categoria,
        "fecha_ipo": perfil.get("ipo"),
    })

df_perfiles = pd.DataFrame(filas_perfil)
df_perfiles

## 3. Precio actual de cada ticker
Esto es una "foto" del momento — no se guarda tal cual en ninguna tabla histórica, pero sirve para validar que los datos tengan sentido.

In [ ]:
filas_precio = []

for ticker in tickers:
    precio = obtener_precio_actual(ticker)
    filas_precio.append({
        "ticker": ticker,
        "precio_actual": precio.get("c"),
        "apertura": precio.get("o"),
        "maximo": precio.get("h"),
        "minimo": precio.get("l"),
        "cierre_anterior": precio.get("pc"),
    })

df_precios = pd.DataFrame(filas_precio)
df_precios

## 4. Histórico diario (yfinance)
Se trae por separado para cada ticker (esto llena `Cotizacion_Historica`). Aquí mostramos solo AAPL como ejemplo.

In [ ]:
historico_aapl = obtener_historico("AAPL", periodo="3mo")
df_historico = pd.DataFrame(historico_aapl)
df_historico.tail(10)

## 5. (Opcional) Graficar el precio de cierre para revisar visualmente

In [ ]:
df_historico.plot(x="fecha", y="precio_cierre", figsize=(10, 4), title="AAPL - Precio de cierre")

## 6. Traer histórico para TODOS los tickers a la vez
Junta todo en un solo DataFrame, listo para revisar antes de insertarlo en `Cotizacion_Historica`.

In [ ]:
todos_historicos = []

for ticker in tickers:
    filas = obtener_historico(ticker, periodo="3mo")
    for fila in filas:
        fila["ticker"] = ticker
        todos_historicos.append(fila)

df_todos = pd.DataFrame(todos_historicos)
print(f"Total de filas: {len(df_todos)}")
df_todos.head(10)